# 04 — Contact Fatigue Model

METIS buildathon ML training notebook. Uses the same training logic as `train_metis_models.py`.

Fatigue model: multiclass prediction of LOW / MEDIUM / HIGH contact fatigue.

In [ ]:

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, log_loss, roc_auc_score
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
BASE_FEATURES = [
    "amount", "days_overdue", "previous_failed_count",
    "historical_payment_rate", "contact_count",
    "previous_no_response_count", "segment_high_value",
    "segment_at_risk", "failure_insufficient_funds",
]
INTERVENTIONS = ["RETRY", "REMINDER", "PAYMENT_LINK", "PAYMENT_PLAN", "NEGOTIATION"]

ROOT = Path.cwd()
for candidate in [
    ROOT / "recovery_events.csv",
    ROOT / "../recovery_events.csv",
    ROOT / "../data/recovery_events.csv",
    ROOT / "../../ml_data/recovery_events.csv",
]:
    if candidate.exists():
        DATA_PATH = candidate.resolve()
        break
else:
    raise FileNotFoundError("Place recovery_events.csv in the notebook directory, ../, ../data/, or ../../ml_data/")

OUTPUT_DIR = ROOT / "../backend/models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)


In [ ]:

FATIGUE_FEATURES = [
    "contact_count",
    "previous_no_response_count",
    "days_overdue",
    "historical_payment_rate",
]

train_idx, test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df["intervention_type"]
)
train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

model = LGBMClassifier(
    objective="multiclass",
    num_class=3,
    n_estimators=180,
    learning_rate=0.04,
    num_leaves=15,
    min_child_samples=30,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)

model.fit(
    train_df[FATIGUE_FEATURES],
    train_df["fatigue_level"],
    categorical_feature=[]
)

pred = model.predict(test_df[FATIGUE_FEATURES])
proba = model.predict_proba(test_df[FATIGUE_FEATURES])

metrics = {
    "balanced_accuracy": float(
        balanced_accuracy_score(test_df["fatigue_level"], pred)
    ),
    "accuracy": float(
        accuracy_score(test_df["fatigue_level"], pred)
    ),
    "n_test": int(len(test_df)),
    "classes": [str(c) for c in model.classes_],
    "mean_max_probability": float(np.max(proba, axis=1).mean()),
}
metrics


In [ ]:

artifact = {
    "model": model,
    "feature_names": FATIGUE_FEATURES,
    "model_version": "metis-fatigue-lgbm-v1",
}
joblib.dump(artifact, OUTPUT_DIR / "fatigue_model.pkl", compress=3)

with open(OUTPUT_DIR / "fatigue_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved:", OUTPUT_DIR / "fatigue_model.pkl")
print(metrics)
